# CLM-001-QFIRST — Run All

Automatic preflight → assets → GPU/restore checks → paired development → analysis → GitHub return.
Use the existing Notebook and HF_TOKEN / GITHUB_TOKEN Secrets. Failed checks stop development.
Exploratory results only; confirmation remains sealed. Save Version retains full checkpoints.


In [ ]:
from pathlib import Path

SOURCE_REF = "codex/clm001-qfirst-integrated"
ROOT = Path("/kaggle/working/clm001-qfirst-code")
OUTPUT = Path("/kaggle/working/clm001-qfirst-output")
ASSETS = Path("/kaggle/working/clm001-qfirst-assets")

In [ ]:
import importlib.util
import subprocess
import sys


def run(args, **kwargs):
    print("[CLM-001]", " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True, **kwargs)


# Keep Kaggle's installed stack. Install only genuinely missing experiment modules.

requirements = {
    "pytest": "pytest",
    "numpy": "numpy",
    "tokenizers": "tokenizers",
    "huggingface_hub": "huggingface_hub",
    "datasets": "datasets>=2.18,<5",
}
missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    run([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if not (ROOT / ".git").exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init", "--quiet", ROOT])
    run(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], cwd=ROOT)
run(["git", "sparse-checkout", "init", "--no-cone"], cwd=ROOT)
run(
    [
        "git",
        "sparse-checkout",
        "set",
        "--no-cone",
        "/pyproject.toml",
        "/.github/workflows/clm001-qfirst-cpu.yml",
        "/src/",
        "/shared/python/",
        "/tests/research/clm001_qfirst/",
        "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
        "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
        "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
        "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
    ],
    cwd=ROOT,
)
run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_REF], cwd=ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT)
source = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
BASE = ROOT / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated"
OUTPUT = OUTPUT / source[:12]
OUTPUT.mkdir(parents=True, exist_ok=True)
print(
    {
        "experiment": "CLM-001-QFIRST",
        "mode": "auto",
        "source": source,
        "scientific_decision": False,
    },
    flush=True,
)
run(
    [sys.executable, BASE / "run.py", "--mode", "auto", "--output", OUTPUT, "--assets", ASSETS],
    cwd=ROOT,
)
print("Retain with Save Version:", OUTPUT, flush=True)